In [ ]:
!pip install langchain_huggingface
!pip install langchain_core
!pip install hugging_face_hub

In [2]:
import re
import pandas as pd
from typing import Annotated, TypedDict, List, Optional,NotRequired
from datasets import load_dataset
from langgraph.graph import StateGraph, START, END
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langchain_core.prompts import PromptTemplate
from pydantic import BaseModel,Field
from langchain_core.output_parsers import PydanticOutputParser
from huggingface_hub import login
from langgraph.constants import Send
import operator

/tmp/ipykernel_1015/3097693908.py:11: LangGraphDeprecatedSinceV10: Importing Send from langgraph.constants is deprecated. Please use 'from langgraph.types import Send' instead. Deprecated in LangGraph V1.0 to be removed in V2.0.
  from langgraph.constants import Send


In [13]:
login()

In [4]:
class ThinkerState(TypedDict):
    id: int
    question: str
    true_answer: str
    thinker_response: NotRequired[str]
    thinker_cot: NotRequired[str]
    thinker_answer: NotRequired[str]
    is_correct: NotRequired[int]

In [5]:
class ThinkerOutput(BaseModel):
    thinker_cot: str = Field(
        description="Step-by-step reasoning for solving the problem"
    )
    thinker_answer: str = Field(
        description="Final numeric answer only (no explanation)"
    )


thinker_parser = PydanticOutputParser(
    pydantic_object=ThinkerOutput
)

In [6]:
thinker_llm = HuggingFaceEndpoint(
    repo_id="meta-llama/Llama-3.1-8B-Instruct",

    temperature=0.2,
    max_new_tokens=768,
    do_sample=False,
    )
thinker_model = ChatHuggingFace(llm=thinker_llm)

In [7]:
thinker_prompt = PromptTemplate(
    template="""
You are a careful and rigorous math problem solver.

Your task is to solve the problem step by step.

STRICT RULES:
1. First, write detailed step-by-step reasoning.
2. DO NOT reveal the final answer anywhere in the reasoning.
3. The reasoning must NOT contain the final numeric answer.
4. You MUST always provide a final answer.
5. Even if you are unsure or think your reasoning may be incorrect, you MUST still give your best possible final answer.
6. The final answer must be a single number (no units, no explanation).
7. Your response is INVALID if thinker_answer is missing.

OUTPUT FORMAT (STRICT JSON):
{format_instructions}

Problem:
{question}
""",
    input_variables=["question"],
    partial_variables={
        "format_instructions": thinker_parser.get_format_instructions()
    }
)

In [8]:
def normalize(x):
    if x is None:
        return None
    return re.sub(r"[^\d\.-]", "", str(x))


def extract_json(text):
    match = re.search(r"\{.*\}", text, re.DOTALL)
    return match.group(0) if match else None


def is_invalid_answer(ans):
    if ans is None:
        return True
    ans = str(ans).strip().lower()
    return ans in ["", "thinker_answer", "missing"]


def remove_answer_from_cot(cot, answer):
    if cot and answer:
        cot = re.sub(rf"\b{re.escape(str(answer))}\b", "", cot)
    return cot.strip() if cot else cot


def extract_last_number_strong(text):
    tail = text[-200:]
    nums = re.findall(r"[-+]?\d*\.\d+|\d+", tail)
    return nums[-1] if nums else None


def thinker_agent(state: ThinkerState):

    q = state["question"]
    gt = state["true_answer"]

    formatted_prompt = thinker_prompt.format(question=q)
    response = thinker_model.invoke(formatted_prompt)
    if isinstance(response, str):
      raw_text = response.strip()
    else:
      raw_text = response.content.strip()

    cot, answer = None, None

    try:
        json_text = extract_json(raw_text)
        if json_text:
            parsed = thinker_parser.parse(json_text)
            cot = parsed.thinker_cot
            answer = parsed.thinker_answer
    except Exception:
        pass


    candidate = extract_last_number_strong(raw_text)

    if is_invalid_answer(answer):
        match = re.search(r"####\s*([-0-9.,]+)", raw_text)
        if match:
            answer = match.group(1)

    if is_invalid_answer(answer):
        answer = candidate


    if is_invalid_answer(answer):
        answer = candidate

    if cot is None:
        cot = raw_text

    cot = remove_answer_from_cot(cot, answer)

    norm_pred = normalize(answer)
    norm_gt = normalize(gt)

    is_correct = int(norm_pred == norm_gt) if norm_pred and norm_gt else 0

    return {
        "thinker_response": raw_text,
        "thinker_cot": cot,
        "thinker_answer": answer,
        "is_correct": is_correct
    }

In [9]:
dataset = load_dataset("gsm8k", "main")

def extract_answer(example):
    ans = example["answer"]
    match = re.search(r"####\s*([-0-9.,]+)", ans)
    example["final_answer"] = match.group(1) if match else None
    return example

dataset = dataset.map(extract_answer)

README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/2.31M [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/419k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

Map:   0%|          | 0/7473 [00:00<?, ? examples/s]

Map:   0%|          | 0/1319 [00:00<?, ? examples/s]

In [10]:
print(dataset["train"][0]["question"])
print(dataset["train"][0]["final_answer"])

Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?
72


In [11]:
samples = dataset["train"].select(range(3))

In [20]:
for i, example in enumerate(samples):

    state = {
        "id": i+1,
        "question": example["question"],
        "true_answer": example["final_answer"]
    }

    result = thinker_agent(state)

    print("\n" + "="*60)
    print(f"ID: {i+1}")
    print("QUESTION:\n", example["question"])

    print("\nGROUND TRUTH:", example["final_answer"])
    print("THINKER ANSWER:", result.get("thinker_answer"))
    print("IS CORRECT:", result.get("is_correct"))

    print("\n--- COT ---")
    print(result.get("thinker_cot"))

    print("\n--- RAW OUTPUT (truncated) ---")
    print(result.get("thinker_response"))


ID: 1
QUESTION:
 Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natalia sell altogether in April and May?

GROUND TRUTH: 72
THINKER ANSWER: None
IS CORRECT: 0

--- COT ---
To find the total number of clips Natalia sold in April and May, we need to calculate the number of clips she sold in each month and then add them together.

First, we know that Natalia sold clips to 48 of her friends in April. This is the number of clips she sold in April.

Next, we are told that she sold half as many clips in May as she did in April. To find the number of clips she sold in May, we need to divide the number of clips she sold in April by 2.

Once we have the number of clips she sold in May, we can add it to the number of clips she sold in April to find the total number of clips she sold in both months.

We will start by calculating the number of clips she sold in May.

--- RAW OUTPUT (truncated) ---
{
  "thinker_cot": "To find the to

In [21]:
all_results = []


for i, example in enumerate(samples):
    state = {
        "id": i+1,
        "question": example["question"],
        "true_answer": example["final_answer"]
    }

    result = thinker_agent(state)


    record = {**state, **result}
    all_results.append(record)
def create_filtered_csv(results_list, output_filename="thinker_correct.csv"):

    df = pd.DataFrame(results_list)


    df_correct = df[df["is_correct"] == 1]


    df_correct.to_csv(output_filename, index=False)

    print(f"Total questions processed: {len(df)}")
    print(f"Successfully answered by Thinker: {len(df_correct)}")
    print(f"Saved correct responses to {output_filename}\n")

    return df_correct

In [22]:
df_correct = create_filtered_csv(all_results)

filtered_tasks = []
for _, row in df_correct.iterrows():
    filtered_tasks.append({
        "task_id": row["id"],
        "question": row["question"],
        "correct_ans": row["true_answer"],
        "thinker_cot": row["thinker_cot"]
    })

Total questions processed: 3
Successfully answered by Thinker: 2
Saved correct responses to thinker_correct.csv



In [38]:
df_correct

,id,question,true_answer,thinker_response,thinker_cot,thinker_answer,is_correct
1,2,Weng earns $12 an hour for babysitting. Yester...,10,"{\n ""thinker_cot"": ""To find out how much Weng...","To find out how much Weng earned, we need to f...",10,1
2,3,Betty is saving money for a new wallet which c...,5,"{\n ""thinker_cot"": ""To solve this problem, we...","To solve this problem, we need to determine ho...",5,1


In [37]:
from langchain_huggingface import HuggingFacePipeline
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline


model_id = "Qwen/Qwen2.5-1.5B-Instruct"


tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto"
)


pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=256,
    temperature=0.1,
    max_length=None,
    do_sample=False,
    return_full_text=False
)


executor_llm = HuggingFacePipeline(pipeline=pipe)
executor_model = ChatHuggingFace(llm=executor_llm)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'max_length', 'temperature', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [39]:
class ExecutorOutput(BaseModel):
  executor_answer: str = Field(
        description="Final numeric answer only (no explanation)"
    )
executor_parser = PydanticOutputParser(pydantic_object=ExecutorOutput)


In [40]:
independent_prompt = PromptTemplate(
    template="""
You are a rigorous math solver. Solve the problem and provide the final numeric answer.
{format_instructions}

Problem:
{question}
""",
    input_variables=["question"],
    partial_variables={"format_instructions": executor_parser.get_format_instructions()}
)

guided_prompt = PromptTemplate(
    template="""
You are a rigorous math solver. Solve the problem using the provided step-by-step reasoning.
{format_instructions}

Problem:
{question}

Helpful Reasoning:
{thinker_cot}
""",
    input_variables=["question", "thinker_cot"],
    partial_variables={"format_instructions": executor_parser.get_format_instructions()}
)

In [41]:
def extract_executor_answer(text):
    try:
        match = re.search(r"\{.*\}", text, re.DOTALL)
        if match:
            parsed = executor_parser.parse(match.group(0))
            return parsed.executor_answer
    except Exception:
        pass

    nums = re.findall(r"[-+]?\d*\.\d+|\d+", text[-100:])
    return nums[-1] if nums else None

def normalize(x):
    if x is None:
        return None
    return re.sub(r"[^\d\.-]", "", str(x))

In [51]:
class OverallState(TypedDict):
    filtered_tasks: List[dict]
    executor_results: Annotated[List[dict], operator.add]

class ExecutorState(TypedDict):
    task_id: int
    executor_id: int
    question: str
    correct_ans: str
    thinker_cot: str
    independent_ans: Optional[str]
    guided_ans: Optional[str]
    status: Optional[str]
    executor_results: List[dict]

In [59]:
NUM_EXECUTORS = 2
def independent_solve_node(state: ExecutorState):

    formatted_prompt = independent_prompt.format(question=state["question"])
    response = executor_model.invoke(formatted_prompt)
    ind_ans = normalize(extract_executor_answer(response.content))
    return {"independent_ans": ind_ans}

def route_after_independent(state: ExecutorState):

    if state["independent_ans"] == state["correct_ans"]:
        return "success_end"
    return "guided_solve_node"

def guided_solve_node(state: ExecutorState):

    formatted_prompt = guided_prompt.format(
        question=state["question"],
        thinker_cot=state["thinker_cot"]
    )
    response = executor_model.invoke(formatted_prompt)
    guided_ans = normalize(extract_executor_answer(response.content))
    return {"guided_ans": guided_ans}

def evaluate_final_status(state: ExecutorState):

    ind_ans = state.get("independent_ans")
    guided_ans = state.get("guided_ans")
    correct = state.get("correct_ans")

    if ind_ans == correct:
        status = "Independent Success"
    elif guided_ans == correct:
        status = "CoT Rescued"
    else:
        status = "Total Failure"


    final_result = {
        "task_id": state.get("task_id"),
        "executor_id": state.get("executor_id"),
        "correct_ans": correct,
        "status": status,
        "independent_ans": ind_ans,
        "guided_ans": guided_ans
    }
    return {"executor_results": [final_result]}

In [58]:
executor_builder = StateGraph(ExecutorState)
executor_builder.add_node("independent_solve_node", independent_solve_node)
executor_builder.add_node("guided_solve_node", guided_solve_node)
executor_builder.add_node("evaluate_final_status", evaluate_final_status)

executor_builder.add_edge(START, "independent_solve_node")
executor_builder.add_conditional_edges("independent_solve_node", route_after_independent, {
    "success_end": "evaluate_final_status",
    "guided_solve_node": "guided_solve_node"
})
executor_builder.add_edge("guided_solve_node", "evaluate_final_status")
executor_builder.add_edge("evaluate_final_status", END)
executor_graph = executor_builder.compile()

In [ ]:
def trigger_parallel_executors(state: OverallState):

    sends = []
    for task in state["filtered_tasks"]:

        for i in range(1, NUM_EXECUTORS + 1):
            executor_task = task.copy()
            executor_task["executor_id"] = i
            sends.append(Send("executor_graph", executor_task))

    print(f"Fanning out {len(state['filtered_tasks'])} tasks to {NUM_EXECUTORS} executors each (Total parallel runs: {len(sends)})...")
    return sends


main_builder = StateGraph(OverallState)
main_builder.add_node("executor_graph", executor_graph)
main_builder.add_conditional_edges(START, trigger_parallel_executors, ["executor_graph"])
main_builder.add_edge("executor_graph", END)
final_pipeline = main_builder.compile()

In [61]:
filtered_tasks = []
for _, row in df_correct.iterrows():

    filtered_tasks.append({
        "task_id": row["id"],
        "question": row["question"],
        "correct_ans": normalize(row["true_answer"]),
        "thinker_cot": row["thinker_cot"]
    })


if len(filtered_tasks) > 0:
    initial_state = {"filtered_tasks": filtered_tasks, "executor_results": []}


    final_state = final_pipeline.invoke(initial_state)
    results_data = final_state.get("executor_results", [])


    metrics_by_executor = {
        i: {"Independent Success": 0, "CoT Rescued": 0, "Total Failure": 0, "total_tasks": 0}
        for i in range(1, NUM_EXECUTORS + 1)
    }

    for res in results_data:
        e_id = res.get("executor_id")
        status = res.get("status")


        if e_id and e_id in metrics_by_executor:
            metrics_by_executor[e_id][status] += 1
            metrics_by_executor[e_id]["total_tasks"] += 1


    print("\n" + "="*60)
    print("FINAL REUSABILITY METRICS (PER EXECUTOR)")
    print("="*60)

    for e_id in range(1, NUM_EXECUTORS + 1):
        stats = metrics_by_executor[e_id]
        ind_success = stats["Independent Success"]
        cot_rescued = stats["CoT Rescued"]
        total_fail = stats["Total Failure"]
        total = stats["total_tasks"]

        initial_failures = cot_rescued + total_fail
        reusability = (cot_rescued / initial_failures * 100) if initial_failures > 0 else 0.0

        print(f"\n[ EXECUTOR {e_id} ] - Processed {total} tasks")
        print(f"  ├─ Independent Successes : {ind_success}")
        print(f"  ├─ CoT Rescued           : {cot_rescued}")
        print(f"  ├─ Total Failures        : {total_fail}")

        if initial_failures > 0:
            print(f"  └─ Reusability Score     : {reusability:.2f}%")
        else:
            print(f"  └─ Reusability Score     : N/A (No initial failures)")

else:
    print("Pipeline halted: No correct Thinker answers available.")



Fanning out 2 tasks to 2 executors each (Total parallel runs: 4)...

FINAL REUSABILITY METRICS (PER EXECUTOR)

[ EXECUTOR 1 ] - Processed 2 tasks
  ├─ Independent Successes : 0
  ├─ CoT Rescued           : 2
  ├─ Total Failures        : 0
  └─ Reusability Score     : 100.00%

[ EXECUTOR 2 ] - Processed 2 tasks
  ├─ Independent Successes : 0
  ├─ CoT Rescued           : 2
  ├─ Total Failures        : 0
  └─ Reusability Score     : 100.00%
